# Olist Orders – Data Preprocessing

## Objective

Prepare the Olist orders dataset for the Silver layer.

The preprocessing focuses on data quality validation, temporal consistency, business logic, feature engineering and reproducible quality checks.

## Granularity

One row represents one `order_id` record.

## Key Definitions

The `orders` dataset uses the following key structure:

- `id_orders`: technical unique key of the Silver layer, generated sequentially starting at 1.
- `order_id`: unique identifier of an order.
- `customer_id`: identifier linking an order to a customer record.
- `order_status`: current status of the order within the order lifecycle.
- Order timestamps: timestamps describing the main stages of the order lifecycle, from purchase through approval, carrier handover and customer delivery.
- The unique key is introduced in the Silver layer, while the original `order_id` is preserved as the business key.

## Processing Steps

1. Load raw data
2. Perform data audit
3. Validate keys and customer references
4. Validate order status and timestamp patterns
5. Check temporal consistency
6. Create temporal quality flags
7. Engineer business features
8. Validate engineered features and quality flags
9. Run final data quality checks
10. Export the Silver dataset

## 1. Setup

Import required Python libraries and define project paths.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_PATH = Path("../datasets/raw")
PROCESSED_PATH = Path("../datasets/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
bronze_orders = pd.read_csv(RAW_PATH / "olist_orders_dataset.csv")
bronze_customers = pd.read_csv(RAW_PATH / "olist_customers_dataset.csv")

orders_raw_row_count = len(bronze_orders)
customers_raw_row_count = len(bronze_customers)

silver_customers = pd.read_csv(PROCESSED_PATH / "silver_customers.csv")

print(f"Bronze orders: {bronze_orders.shape}")
print(f"Bronze customers: {bronze_customers.shape}")

bronze_orders.head()

Bronze orders: (99441, 8)
Bronze customers: (99441, 5)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:

print("Shape:", bronze_orders.shape)
print("\nColumns:")
print(bronze_orders.columns.tolist())

print("\nData types:")
print(bronze_orders.dtypes)

print("\nFirst rows:")
display(bronze_orders.head())

print("\nMissing values:")
display(
    bronze_orders.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
    .assign(
        missing_pct=lambda x: (x["missing_count"] / len(bronze_orders) * 100).round(2)
    )
)

print("\nDuplicate rows:", bronze_orders.duplicated().sum())

Shape: (99441, 8)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

First rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Missing values:


,missing_count,missing_pct
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16
order_id,0,0.00
order_purchase_timestamp,0,0.00
order_status,0,0.00
customer_id,0,0.00
order_estimated_delivery_date,0,0.00



Duplicate rows: 0


## 2. Data Quality & Key Validation

Validate primary key uniqueness, customer references and order status values.

### 2.1 Key Integrity

Validate the uniqueness of the order primary key and identify duplicate order records.

#### Key Definition

The `orders` dataset uses the following key structure:

- `id_orders`: technical surrogate key of the Silver layer, generated sequentially starting at 1.
- `order_id`: business/natural key identifying the original Olist order.
- `customer_id`: foreign key referencing `customer_id` in the customers dataset.

The surrogate key is introduced in the Silver layer, while the original `order_id` is preserved as the business key.

In [5]:
# Validate primary key uniqueness

print("Unique order_ids:", bronze_orders["order_id"].nunique())
print("Total rows:", len(bronze_orders))
print("Duplicate order_ids:", bronze_orders["order_id"].duplicated().sum())

customer_id_match = bronze_orders["customer_id"].isin(bronze_customers["customer_id"])

print("Orders with valid customer_id:", customer_id_match.sum())
print("Orders with missing customer reference:", (~customer_id_match).sum())


Unique order_ids: 99441
Total rows: 99441
Duplicate order_ids: 0
Orders with valid customer_id: 99441
Orders with missing customer reference: 0


### 2.2 Order Status Distribution

Examine the distribution of order statuses to understand the composition of the order dataset and identify dominant order lifecycle stages.

In [6]:
print("Order status distribution:")
display(
    bronze_orders["order_status"]
    .value_counts()
    .to_frame("order_count")
    .assign(percentage=lambda x: (x["order_count"] / len(bronze_orders) * 100).round(2)))

Order status distribution:


,order_count,percentage
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


### 2.3 Temporal Consistency

Validate the logical sequence of order events and identify records with impossible timestamp relationships.

##### Temporal Fields Preparation

Convert order lifecycle timestamps to datetime format before validating their temporal consistency.

In [7]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"]

bronze_orders[date_columns] = bronze_orders[date_columns].apply(pd.to_datetime)

bronze_orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [8]:
print("Approved before purchase:",(bronze_orders["order_approved_at"] < bronze_orders["order_purchase_timestamp"]).sum())

print("Carrier before purchase:",(bronze_orders["order_delivered_carrier_date"] < bronze_orders["order_purchase_timestamp"]).sum())

print("Customer delivery before purchase:",(bronze_orders["order_delivered_customer_date"] < bronze_orders["order_purchase_timestamp"]).sum())

print("Customer delivery before carrier:",(bronze_orders["order_delivered_customer_date"]< bronze_orders["order_delivered_carrier_date"]).sum())

Approved before purchase: 0
Carrier before purchase: 166
Customer delivery before purchase: 0
Customer delivery before carrier: 23


In [9]:
# Missing timestamp rates by order status

missing_pct_by_status = (bronze_orders.groupby("order_status")[date_columns].apply(lambda x: x.isna().mean() * 100).round(2))

display(missing_pct_by_status)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0.0,0.00,100.0,100.00,0.0
canceled,0.0,22.56,88.0,99.04,0.0
created,0.0,100.00,100.0,100.00,0.0
delivered,0.0,0.01,0.0,0.01,0.0
invoiced,0.0,0.00,100.0,100.00,0.0
processing,0.0,0.00,100.0,100.00,0.0
shipped,0.0,0.00,0.0,100.00,0.0
unavailable,0.0,0.00,100.0,100.00,0.0


### 2.4 Temporal Anomaly Assessment

Quantify the magnitude of temporal inconsistencies to distinguish isolated data-quality issues from potentially systematic problems.

In [10]:
# Assess magnitude of temporal anomalies

# Carrier date before purchase
carrier_anomalies = bronze_orders.loc[
    bronze_orders["order_delivered_carrier_date"]
    < bronze_orders["order_purchase_timestamp"]
].copy()

carrier_anomalies["difference_minutes"] = (
    carrier_anomalies["order_purchase_timestamp"]
    - carrier_anomalies["order_delivered_carrier_date"]
).dt.total_seconds() / 60


# Customer delivery before carrier
delivery_anomalies = bronze_orders.loc[
    bronze_orders["order_delivered_customer_date"]
    < bronze_orders["order_delivered_carrier_date"]
].copy()

delivery_anomalies["difference_minutes"] = (
    delivery_anomalies["order_delivered_carrier_date"]
    - delivery_anomalies["order_delivered_customer_date"]
).dt.total_seconds() / 60


# Summarize anomaly magnitude
print("Carrier before purchase:")
display(
    carrier_anomalies["difference_minutes"].describe()
)

print("\nDelivery before carrier:")
display(
    delivery_anomalies["difference_minutes"].describe()
)

Carrier before purchase:


count       166.000000
mean       1562.368675
std       19135.430706
min           0.400000
25%          14.066667
50%          35.991667
75%          61.050000
max      246545.883333
Name: difference_minutes, dtype: float64


Delivery before carrier:


count       23.000000
mean      4707.307971
std       5358.484182
min         23.300000
25%       1400.300000
50%       2391.866667
75%       7725.541667
max      23178.483333
Name: difference_minutes, dtype: float64

In [11]:
# Review most extreme temporal anomalies

print("Carrier before purchase - largest deviations:")
display(
    carrier_anomalies[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "difference_minutes"
        ]
    ]
    .sort_values("difference_minutes", ascending=False)
    .head(5)
)

print("\nDelivery before carrier - largest deviations:")
display(
    delivery_anomalies[
        [
            "order_id",
            "order_status",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "difference_minutes"
        ]
    ]
    .sort_values("difference_minutes", ascending=False)
    .head(5)
)

Carrier before purchase - largest deviations:


,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,difference_minutes
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-01-26 13:35:00,246545.883333
83321,4021cd7611d6d9ce5ffcd24817fc374f,delivered,2018-08-18 11:49:40,2018-08-14 06:22:00,6087.666667
67844,db090a16182b263b1e896bb26c6f66cf,delivered,2018-07-13 16:14:08,2018-07-13 13:59:00,135.133333
65452,9711d975b961355b4b5d636857e48498,delivered,2018-06-13 15:23:50,2018-06-13 13:15:00,128.833333
79401,89d32b64af005178b318f76cd60f2c3c,delivered,2018-07-06 11:54:40,2018-07-06 09:48:00,126.666667



Delivery before carrier - largest deviations:


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,difference_minutes
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-03-02 17:34:26,2017-02-14 15:15:57,23178.483333
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01,11345.700000
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-06-09 15:07:29,2017-06-02 11:09:16,10318.216667
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10,8654.333333
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-26 11:43:06,2016-10-20 18:03:17,8259.816667


### 2.5 Data Quality Decision

Temporal inconsistencies affect a small number of records. Since the correct timestamps cannot be reliably inferred, the original values are retained.

Affected records are flagged and excluded from derived duration calculations where the affected timestamp is required.

### 2.6 Missing Value Assessment & Treatment

Missing timestamps are largely consistent with the respective order status. For example, delivery-related timestamps are naturally absent for orders that have not reached the delivery stage.

A small number of delivered orders also contain missing timestamps. Since the correct event times cannot be reliably inferred, missing values are retained as `NaN` and no imputation or row deletion is performed.

In [12]:
# Missing values by order status and date column

missing_pct_by_status = (
    bronze_orders
    .groupby("order_status")[date_columns]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2))

display(missing_pct_by_status)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0.0,0.00,100.0,100.00,0.0
canceled,0.0,22.56,88.0,99.04,0.0
created,0.0,100.00,100.0,100.00,0.0
delivered,0.0,0.01,0.0,0.01,0.0
invoiced,0.0,0.00,100.0,100.00,0.0
processing,0.0,0.00,100.0,100.00,0.0
shipped,0.0,0.00,0.0,100.00,0.0
unavailable,0.0,0.00,100.0,100.00,0.0


In [13]:
# Investigate missing timestamps in delivered orders

delivered_missing_dates = bronze_orders[
    (bronze_orders["order_status"] == "delivered")
    & (
        bronze_orders[date_columns]
        .isna()
        .any(axis=1)
    )
].copy()

print("Delivered orders with missing timestamps:",len(delivered_missing_dates))

display(delivered_missing_dates[["order_id", "order_status"] + date_columns].head(5))

Delivered orders with missing timestamps: 23


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16


### 2.7 Data Quality Summary

The order data is suitable for further analysis after applying explicit quality flags. Temporal inconsistencies are retained in the original data but excluded from affected duration calculations. Missing timestamps are retained as `NaN` where no reliable imputation is possible.

The resulting dataset preserves the original observations while preventing identified data-quality issues from distorting derived analytical features.

## 3. Temporal Quality Flags

Create flags for temporal inconsistencies without modifying the original timestamp columns.

### 3.1 Temporal anomaly flags

In [14]:
# Create temporal quality flags

bronze_orders["carrier_before_purchase_flag"] = (bronze_orders["order_delivered_carrier_date"]<bronze_orders["order_purchase_timestamp"])

bronze_orders["delivery_before_carrier_flag"] = (bronze_orders["order_delivered_customer_date"]<bronze_orders["order_delivered_carrier_date"])

### 3.2 Missing timestamp flags

In [15]:
# Create missing timestamp flags

bronze_orders["missing_approval_flag"] = (bronze_orders["order_approved_at"].isna())

bronze_orders["missing_carrier_date_flag"] = (bronze_orders["order_delivered_carrier_date"].isna())

bronze_orders["missing_delivery_date_flag"] = (bronze_orders["order_delivered_customer_date"].isna())

### 3.3 Flag validation

In [16]:
# Validate temporal quality flags

print("Carrier before purchase:",bronze_orders["carrier_before_purchase_flag"].sum())

print("Delivery before carrier:",bronze_orders["delivery_before_carrier_flag"].sum())

print("Missing approval timestamp:",bronze_orders["missing_approval_flag"].sum())

print("Missing carrier timestamp:",bronze_orders["missing_carrier_date_flag"].sum())

print("Missing customer delivery timestamp:",bronze_orders["missing_delivery_date_flag"].sum())

Carrier before purchase: 166
Delivery before carrier: 23
Missing approval timestamp: 160
Missing carrier timestamp: 1783
Missing customer delivery timestamp: 2965


### 3.4 Quality flag summary

In [17]:
# Summary of quality flags

quality_flag_summary = pd.DataFrame({
    "flag": [
        "carrier_before_purchase_flag",
        "delivery_before_carrier_flag",
        "missing_approval_flag",
        "missing_carrier_date_flag",
        "missing_delivery_date_flag"],
    "count": [
        bronze_orders["carrier_before_purchase_flag"].sum(),
        bronze_orders["delivery_before_carrier_flag"].sum(),
        bronze_orders["missing_approval_flag"].sum(),
        bronze_orders["missing_carrier_date_flag"].sum(),
        bronze_orders["missing_delivery_date_flag"].sum()]})

quality_flag_summary["percentage"] = (quality_flag_summary["count"] / len(bronze_orders) * 100).round(3)

display(quality_flag_summary)

,flag,count,percentage
0,carrier_before_purchase_flag,166,0.167
1,delivery_before_carrier_flag,23,0.023
2,missing_approval_flag,160,0.161
3,missing_carrier_date_flag,1783,1.793
4,missing_delivery_date_flag,2965,2.982


# 4. Silver Data Validation

Create the Silver orders dataset from the validated Bronze orders data.

The Silver layer preserves the original order-level data and adds the technical surrogate key and data-quality flags.

In [20]:
# Creating Silver Datasets

silver_orders = bronze_orders.copy()

# Create technical surrogate key
silver_orders.insert(
    0,
    "id_orders",
    range(1, len(silver_orders) + 1)
)

# Expected Silver schema
expected_silver_columns = [
    "id_orders",
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "carrier_before_purchase_flag",
    "delivery_before_carrier_flag",
    "missing_approval_flag",
    "missing_carrier_date_flag",
    "missing_delivery_date_flag"
]

silver_orders = silver_orders[expected_silver_columns]

# Validate row count against raw dataset
assert len(silver_orders) == orders_raw_row_count

# Validate silver unique key
assert silver_orders["id_orders"].notna().all()
assert silver_orders["id_orders"].is_unique
assert silver_orders["id_orders"].min() == 1
assert silver_orders["id_orders"].max() == len(silver_orders)


# Validate primary key
assert silver_orders["order_id"].is_unique


# Reorder columns according to the Silver schema
silver_orders = silver_orders[expected_silver_columns]


# Validate expected columns
assert list(silver_orders.columns) == expected_silver_columns


# Validate required key fields
assert silver_orders["order_id"].notna().all()
assert silver_orders["customer_id"].notna().all()
assert silver_orders["order_status"].notna().all()
assert silver_orders["order_purchase_timestamp"].notna().all()


# Validate temporal data types
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

assert all(
    pd.api.types.is_datetime64_any_dtype(silver_orders[column])
    for column in date_columns
)


# Validate quality flag data types
quality_flags = [
    "carrier_before_purchase_flag",
    "delivery_before_carrier_flag",
    "missing_approval_flag",
    "missing_carrier_date_flag",
    "missing_delivery_date_flag"
]

assert all(
    silver_orders[column].dtype == bool
    for column in quality_flags
)


# Validate temporal anomaly flags
assert (
    (
        silver_orders["carrier_before_purchase_flag"]
        != (
            silver_orders["order_delivered_carrier_date"]
            < silver_orders["order_purchase_timestamp"]
        )
    ).sum()
    == 0
)

assert (
    (
        silver_orders["delivery_before_carrier_flag"]
        != (
            silver_orders["order_delivered_customer_date"]
            < silver_orders["order_delivered_carrier_date"]
        )
    ).sum()
    == 0
)


# Validate missing timestamp flags
assert (
    (
        silver_orders["missing_approval_flag"]
        != silver_orders["order_approved_at"].isna()
    ).sum()
    == 0
)

assert (
    (
        silver_orders["missing_carrier_date_flag"]
        != silver_orders["order_delivered_carrier_date"].isna()
    ).sum()
    == 0
)

assert (
    (
        silver_orders["missing_delivery_date_flag"]
        != silver_orders["order_delivered_customer_date"].isna()
    ).sum()
    == 0
)


# Create Silver quality summary
silver_quality_summary = (
    silver_orders[quality_flags]
    .sum()
    .to_frame("flagged_count")
)

silver_quality_summary["flagged_pct"] = (
    silver_quality_summary["flagged_count"]
    / len(silver_orders)
    * 100
).round(3)


# Final Silver validation
assert len(silver_orders) == orders_raw_row_count
assert silver_orders["order_id"].is_unique
assert list(silver_orders.columns) == expected_silver_columns


print("Silver data validation passed.")
print(f"Rows: {len(silver_orders):,}")
print(f"Unique order_ids: {silver_orders['order_id'].nunique():,}")
print(f"Columns: {len(silver_orders.columns)}")
print("Silver schema column order:")
print(list(silver_orders.columns))

Silver data validation passed.
Rows: 99,441
Unique order_ids: 99,441
Columns: 14
Silver schema column order:
['id_orders', 'order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'carrier_before_purchase_flag', 'delivery_before_carrier_flag', 'missing_approval_flag', 'missing_carrier_date_flag', 'missing_delivery_date_flag']


In [ ]:
character_counts = (
    silver_orders["order_status"]
    .dropna()
    .astype(str)
    .str.cat(sep="")
)

character_counts = pd.Series(list(character_counts)).value_counts().sort_index()

for character, count in character_counts.items():
    display_character = "<SPACE>" if character == " " else character
    print(f"{display_character}: {count}")

a: 2459
b: 609
c: 1870
d: 195009
e: 293027
g: 301
h: 1107
i: 99123
l: 98321
n: 1849
o: 617
p: 2519
r: 96786
s: 1709
t: 5
u: 609
v: 97403


In [ ]:
order_status_counts = (
    silver_orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="count")
)

order_status_counts

,order_status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [ ]:
length_analysis = []

for column in silver_orders.columns:
    values = silver_orders[column].dropna().astype(str)
    lengths = values.str.len()

    length_analysis.append({
        "column": column,
        "data_type": str(silver_orders[column].dtype),
        "min_length": lengths.min(),
        "max_length": lengths.max(),
        "shortest_value": values.loc[lengths.idxmin()],
        "longest_value": values.loc[lengths.idxmax()]
    })

length_analysis = pd.DataFrame(length_analysis)

length_analysis

,column,data_type,min_length,max_length,shortest_value,longest_value
0,id_orders,int64,1,5,1,10000
1,order_id,str,32,32,e481f51cbdc54678b7cc49136f2d6af7,e481f51cbdc54678b7cc49136f2d6af7
2,customer_id,str,32,32,9ef432eb6251297304e76186b10a928d,9ef432eb6251297304e76186b10a928d
3,customer_unique_id,str,32,32,7c396fd4830fd04220f754e42b4e5bff,7c396fd4830fd04220f754e42b4e5bff
4,order_status,str,7,11,shipped,unavailable
5,order_purchase_timestamp,datetime64[us],19,19,2017-10-02 10:56:33,2017-10-02 10:56:33
6,order_approved_at,datetime64[us],19,19,2017-10-02 11:07:15,2017-10-02 11:07:15
7,order_delivered_carrier_date,datetime64[us],19,19,2017-10-04 19:55:00,2017-10-04 19:55:00
8,order_delivered_customer_date,datetime64[us],19,19,2017-10-10 21:25:13,2017-10-10 21:25:13
9,order_estimated_delivery_date,datetime64[us],10,10,2017-10-18,2017-10-18


## 6. Silver Dataset Export

After successfully passing all data quality checks, the cleaned customer
dataset is exported to the processed data layer.

The raw dataset remains unchanged.

In [ ]:
output_path = PROCESSED_PATH / "silver_orders.csv"

silver_orders.to_csv(
    output_path,
    index=False
)

print(f"Silver dataset exported to: {output_path}")

Silver dataset exported to: ..\datasets\processed\silver_orders.csv


# 7. Orders Dataset Summary 

The resulting Silver dataset preserves the original order-level observations and includes the customer mapping, technical surrogate key and explicit data-quality indicators.

The `customer_unique_id` was added by mapping `customer_id` from the orders dataset to the validated Silver customers dataset.

A technical surrogate key `id_orders` was generated for the Silver orders dataset. The key starts at 1 and provides a unique identifier for each order record.

The original `order_id` remains the business key and was preserved unchanged.

Business features and analytical KPIs are intentionally not included in the Silver layer. These will be created in the separate Gold layer based on the validated Silver data.

The Silver dataset is ready for downstream data modeling, feature engineering and business analysis.